# Sampling variability and the Central Limit Theorem

Core stage, lesson C6. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** draw seeded random samples, build the sampling distribution of the mean order value, tell the standard error from the standard deviation, and see the Central Limit Theorem at work.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)

# One number for each order: the sum of its lines, quantity × unit price − discount.
# All 1,800 orders count: the canceled order and order 2075 (the line with the extra zero) are in.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"])
orders["value"] = orders["order_id"].map(lines.groupby("order_id")["amount"].sum())
ordered = pd.to_datetime(orders["ordered_at"])
orders["hour"] = ordered.dt.hour
orders["weekday"] = ordered.dt.day_name()
assert orders["value"].notna().all() and round(orders["value"].sum(), 2) == 22754.25
print(f"Data loaded: {len(orders):,} orders, {orders['value'].mean():.2f} dollars on average, {orders['value'].sum():,.2f} dollars in all.")
# The population of this lesson: the value of each of the 1,800 orders, as an array.
values = orders["value"].to_numpy()
mu = values.mean()

## Unit 1: Seeded random draws

**Predict first.** Two draws of 30 orders follow each other. Will their means be equal? Then the generator starts again from the same seed: what will the third mean be?

In [ ]:
rng = np.random.default_rng(42)         # a random number generator; 42 is its seed
first = rng.choice(values, size=30)     # 30 orders drawn at random; an order can come up twice
second = rng.choice(values, size=30)    # the next 30
print(round(float(first.mean()), 2), round(float(second.mean()), 2))
rng = np.random.default_rng(42)         # the same seed again
print(round(float(rng.choice(values, size=30).mean()), 2))

## Unit 2: The sampling distribution of the mean

**Predict first.** Each of the 1,000 values is the mean of 30 orders. Where will they center, how wide will they spread (single orders have an SD of about 8.6), and will the histogram be a symmetric bell or lopsided?

In [ ]:
rng = np.random.default_rng(42)
means30 = np.array([rng.choice(values, size=30).mean() for _ in range(1000)])   # 1,000 samples, one mean for each
print(round(float(means30.mean()), 2), round(float(means30.std(ddof=1)), 2))
print(round(float(means30.min()), 2), round(float(means30.max()), 2))
edges = np.arange(8, 22.1, 1.0)                                                 # the bars: 1 dollar wide
counts, _ = np.histogram(means30, bins=edges)
print(counts.tolist())
fig, ax = plt.subplots()
ax.hist(means30, bins=edges)
ax.axvline(mu, color="black", linestyle="--")                                   # the mean of all 1,800 orders
ax.set_xlabel("mean of one sample of 30 orders (USD)")
ax.set_ylabel("number of samples")

## Unit 3: Standard error and standard deviation

**Predict first.** The cell prints the SD of single orders two ways, dividing by n and by n - 1, then the SD of the 1,000 sample means for samples of 30, 120 and 480 orders, and last the same for 480 orders drawn without replacement. How will the SD of the means change from one size to the next? Will the last number be larger or smaller than the one before it?

In [ ]:
print(round(float(values.std()), 2), round(float(values.std(ddof=1)), 2))   # the SD of single orders: dividing by n, then by n - 1
rng = np.random.default_rng(42)
for n in (30, 120, 480):
    means = np.array([rng.choice(values, size=n).mean() for _ in range(1000)])
    print(n, round(float(means.std(ddof=1)), 2))
without = np.array([rng.choice(values, size=480, replace=False).mean() for _ in range(1000)])   # no order twice
print("480 without replacement", round(float(without.std(ddof=1)), 2))

**Predict first.** The first line prints sigma divided by the square root of n for the three sizes. In real work you have one sample, not 1,000. From the first sample of 30 of unit 1, will s divided by the square root of 30 land near the 1.51 that the 1,000 samples gave?

In [ ]:
sigma = values.std()                    # the SD of all 1,800 orders, dividing by n
print([round(float(sigma / np.sqrt(n)), 2) for n in (30, 120, 480)])
rng = np.random.default_rng(42)
one = rng.choice(values, size=30)     # the first sample of unit 1
s = one.std(ddof=1)                   # the SD of this one sample, dividing by n - 1
print(round(float(s), 2), round(float(s / np.sqrt(30)), 2))

## Unit 4: The Central Limit Theorem

**Predict first.** The skewness of the orders themselves is printed first. As the sample size grows from 30 to 120 to 480, will the skewness of the sample means rise, fall or stay near that number?

In [ ]:
print(round(float(stats.skew(values)), 2))
rng = np.random.default_rng(42)
for n in (30, 120, 480):
    means = np.array([rng.choice(values, size=n).mean() for _ in range(1000)])
    print(n, round(float(stats.skew(means)), 2))

## Finish it

### Exercise 1: Finish it

**Task:** How much would the mean order value of 100 Harbor orders move from sample to sample? Take the 355 Harbor orders as the population. Find the standard deviation of that population, then the standard error of the mean for samples of 100 orders, rounded to 2 decimals.

**Expected:** `answer_1` is a number.

In [ ]:
harbor = orders[orders["store_id"] == 3]["value"].to_numpy()  # given: the 355 Harbor orders as the population
# the SD of this population, then the standard error of the mean for samples of 100, rounded to 2 decimals
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 0.75, traps={7.51: 'That is the SD of single orders. The standard error of a mean divides it by the square root of the sample size.'})

<details>
<summary>Hint</summary>

Unit 3 divided the SD of the population by the square root of the sample size. `harbor.std()` is the SD of the 355 orders.

</details>

<details>
<summary>Solution</summary>

```python
harbor = orders[orders["store_id"] == 3]["value"].to_numpy()
sigma_harbor = harbor.std()
answer_1 = round(float(sigma_harbor / np.sqrt(100)), 2)
```

0.75: Harbor's orders have an SD of 7.51, and a mean of 100 of them has a standard error of 7.51 divided by 10. The SD of 7.51 is the spread of single orders, so it is not the answer, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many orders must a sample hold before the standard error of its mean order value is at most 0.5 dollars? Before you start, say in one sentence what the result looks like: one whole number. Use the SD of all 1,800 orders, dividing by n, as sigma, and find the smallest whole sample size that gets the standard error to 0.5 or below. Put it in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 297, traps={18: 'That is sigma divided by 0.5. The standard error shrinks with the square root of n, so n is that number squared.'})

<details>
<summary>Hint</summary>

Turn the formula around: standard error = sigma / sqrt(n), so sqrt(n) = sigma / 0.5. Then ask what n is, and round up with `np.ceil`, because a sample cannot hold a fraction of an order.

</details>

<details>
<summary>Solution</summary>

```python
sigma = values.std()
answer_2 = int(np.ceil((sigma / 0.5) ** 2))
```

297 orders: sigma / 0.5 is 17.226 and its square is 296.75, which rounds up to 297. With 296 orders the standard error is 0.5006, just over the limit. Halving the standard error again would take four times as many orders. The check cell reports 18, the answer without the square, as a hint.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague says that the median of 100 orders moves from sample to sample by about the same amount as the mean does, and gives its standard error as sigma divided by the square root of 100. Check the claim by simulation: draw 1,000 samples of 100 orders with seed 7, take the median of each sample, and set `answer_3` to the SD of the 1,000 medians (dividing by n - 1), rounded to 2 decimals. Fix the lines.

**Expected:** `answer_3` is close to `1.07`: the check cell accepts a difference of up to 0.1, because the number depends on the seed.

In [ ]:
rng = np.random.default_rng(7)
sigma = values.std()
answer_3 = round(float(sigma / np.sqrt(100)), 2)  # the colleague's standard error of the median

In [ ]:
_ = check("Exercise 3", "answer_3", 1.07, traps={0.86: 'That is sigma divided by the square root of 100, the standard error of a mean. Does the median of 100 orders move by the same amount? Simulate it.'}, tol=0.1)

<details>
<summary>Hint</summary>

The standard error of a statistic is the SD of its sampling distribution. Build that distribution for the median, as unit 2 did for the mean: 1,000 samples of 100 orders, one median for each.

</details>

<details>
<summary>Solution</summary>

```python
rng = np.random.default_rng(7)
medians_3 = np.array([np.median(rng.choice(values, size=100)) for _ in range(1000)])
answer_3 = round(float(medians_3.std(ddof=1)), 2)
```

With seed 7 the notebook prints 1.07. The formula the colleague used, 0.86, belongs to the mean; the median of these orders moves more, so its standard error is larger. The check cell reports 0.86.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")